In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from operator import itemgetter
from scipy.optimize import minimize_scalar
from dataclasses import dataclass
import scipy.stats as stats
from functools import partial

In [ ]:
def split_pots(p, data):
    split_idx = round(p * len(data))
    return data[:split_idx], data[split_idx:]

In [ ]:
@dataclass
class Student:
    is_disadv: bool
    true_potential: float
    perceived_potential: float
    assignment_without_bias: float | None = None

In [ ]:
def match_and_store(students_with_scores, attr_name):
    """
    Given a list of (student, score), computes the matching and sets the matching into student.attr_name for each
    
    Usage:
    ```py3
    students_w_scores = [(student, student.true_potential) for student in students]
    match_and_store(students_w_scores, "assignment_without_bias")
    ```
    """
    students_with_scores.sort(key=itemgetter(1), reverse=True)
    n = len(students_with_scores)
    for ix, (st, sc) in enumerate(students_with_scores):
        setattr(st, attr_name, ix/n)

In [ ]:
def compute_pauc(students, lo, hi):
    """
    Given a list of students and lo/hi values, computes PAUC under this debiasing
    """
    students_with_scores = [(st, st.true_potential if lo <= st.perceived_potential <= hi else st.perceived_potential) for st in students]
    students_with_scores.sort(key=itemgetter(1), reverse=True)
    n = len(students_with_scores)
    total_mistreatment = 0.
    c = 0
    for ix, (st, sc) in enumerate(students_with_scores):
        if st.is_disadv:
            total_mistreatment += np.maximum(ix/n - st.assignment_without_bias, 0.)
            c += 1
    return total_mistreatment / c

In [ ]:
def compute_dspps(students):
    # disadv_students_perceived_potentials_sorted
    dspps = np.sort([st.perceived_potential for st in students if st.is_disadv])
    assert np.min(dspps) == dspps[0]
    assert np.max(dspps) == dspps[-1]
    assert np.all(np.diff(dspps)>=0)
    return dspps

In [ ]:
class TooHighBudgetException(Exception):
    pass

In [ ]:
def compute_hi(students, lo, budget):
    dspps = compute_dspps(students)
    c = len(dspps)
    lo_ix = len(dspps[dspps<lo])
    hi_ix = int(lo_ix + c*budget)
    if hi_ix >= c:
        raise TooHighBudgetException("budget too large")
    hi = dspps[hi_ix]
    actual_budget = np.mean((lo<dspps)&(dspps<hi))
    assert np.abs(actual_budget-budget) < 1e-3, f"{actual_budget=} vs {budget=}"
    return hi

In [ ]:
def compute_lo(students, hi, budget):
    dspps = compute_dspps(students)
    c = len(dspps)
    hi_ix = len(dspps[dspps<hi])
    lo_ix = int(hi_ix - c*budget)
    if lo_ix < 0:
        raise TooHighBudgetException("budget too large")
    lo = dspps[lo_ix]
    actual_budget = np.mean((lo<dspps)&(dspps<hi))
    assert np.abs(actual_budget-budget) < 1e-3, f"{actual_budget=} vs {budget=}"
    return lo

In [ ]:
def compute_pauc_given_budget(students, lo, budget):
    try:
        hi = compute_hi(students, lo, budget)
    except TooHighBudgetException:
        return -np.inf
    return compute_pauc(students, lo, hi)

In [ ]:
def compute_bracket(students, budget):
    dspps = compute_dspps(students)
    return dspps[0], dspps[int((1-budget)*len(dspps))//2], dspps[int((1-budget)*len(dspps))]

In [ ]:
def compute_bounds(students, budget):
    dspps = compute_dspps(students)
    return dspps[0], dspps[int((1-budget)*len(dspps))]

In [ ]:
def generate_students(true_pots, p, bias):
    disadv_pots, nondisadv_pots = split_pots(p, true_pots)

    students = [
        Student(
            is_disadv=True,
            true_potential=true_pot,
            perceived_potential=bias(true_pot)
        ) for true_pot in disadv_pots
    ] + [
        Student(
            is_disadv=False,
            true_potential=true_pot,
            perceived_potential=true_pot
        ) for true_pot in nondisadv_pots
    ]

    match_and_store([(st, st.true_potential) for st in students], "assignment_without_bias")
    
    return students

In [ ]:
def compute_theorem_interval(p, beta, alpha, c_hat):
    assert p<1-beta**alpha
    assert p<.5
    threshold = ((1 - p) * (1 - (beta**alpha))) / (2 - p - (beta**alpha) - (p * (beta**alpha)) + (p * (beta**(2 * alpha))))
    if c_hat >= threshold:
        gut = ((1 - p) * (1 - c_hat)) / (p * (beta**alpha) + (1 / (beta**alpha)) - 2 * p)
    else:
        gut = ((p * (beta**alpha) - 1) * c_hat + (1 - p)) / ((1 - p) * (1 / (beta**alpha)))
    return beta*(gut+c_hat)**(-1/alpha), beta*gut**(-1/alpha)

In [ ]:
def bias_func_0(theta, param):
    return param * theta

def bias_func_1(theta, param):
    return theta - param

def bias_func_2(theta, param):
    return (param + np.random.normal(0, 0.02)) * theta

def bias_func_3(theta, param):
    return (param + np.random.uniform(-0.05, 0.05)) * theta

def bias_func_4(theta, param):
    return theta - param + np.random.normal(0, 0.1)

def bias_func_5(theta, param):
    return param * theta + np.random.normal(0, 0.1)

def bias_func_6(theta, param):
    return (param + np.random.uniform(-0.15, 0.15)) * theta

def bias_func_7(theta, param):
    return param * theta + np.random.uniform(-0.3, 0.3)

def bias_func_8(theta, param):
    return (param + np.random.uniform(-0.3, 0.3)) * theta

bias_funcs = [bias_func_0, bias_func_1, bias_func_2, bias_func_3, bias_func_4, bias_func_5, bias_func_6, bias_func_7, bias_func_8]

In [ ]:
alpha = 9
p = .3
model_beta = 0.88

In [ ]:
def sample_true_pot():
    return np.random.pareto(alpha) + 1

In [ ]:
np.random.seed(0)

In [ ]:
STUDENTS = 1_000_000

In [ ]:
naive_bias = partial(bias_func_0, param=model_beta)
true_pots = np.array([sample_true_pot() for _ in range(STUDENTS)])

naive_scores = np.vectorize(naive_bias)(true_pots)

In [ ]:
scenarios = []

for bias_func in bias_funcs:
    def func_to_minimize(param):
        np.random.seed(0)
        bias = np.vectorize(partial(bias_func, param=param))
        samples = bias(true_pots)
        return stats.wasserstein_distance(samples, naive_scores)

    bracketing_vals = [(.01, .88, .99), (.01, .2, .99)]
    for bvals in bracketing_vals:
        try:
            params = minimize_scalar(func_to_minimize, bvals, tol=1e-5)
            best_param = params.x
            print(f"{bias_func.__name__}: {best_param=}, wasserstein: {params.fun}")
            scenarios.append((bias_func.__name__, bias_func, best_param))
        except ValueError:
            pass

In [ ]:
bias = partial(bias_func, param=best_param)

In [ ]:
null_bias = partial(bias_func_0, param=.88)

In [ ]:
null_students = generate_students(true_pots, 1, null_bias)

In [ ]:
students = generate_students(true_pots, 1, bias)

In [ ]:
null_pps = np.array([st.perceived_potential for st in null_students])
pps = np.array([st.perceived_potential for st in students])

In [ ]:
null_pps.mean(), null_pps.std()

In [ ]:
pps.mean(), pps.std()

In [ ]:
plt.figure(figsize=(6, 3)) # Set a good figure size for readability

# Plot the histograms
# Use different colors for clarity and specify labels for the legend
plt.hist(null_pps, bins=121, alpha=0.7, density=True, label='Simple Model', color='skyblue')#, edgecolor='gray')
plt.hist(pps, bins=121, alpha=0.7, density=True, label='Correct Model', color='lightcoral')#, edgecolor='gray')

# Set x-axis limits based on your specification
# Calculate the lower bound clearly for readability
lower_x_limit = 0.84845 - 0.3 - 0.01
plt.xlim((lower_x_limit, 1.6))

# Add descriptive labels and title
# plt.xlabel('Disadvantaged student perceived potential distribution')#, fontsize=12)
# plt.ylabel('Frequency', fontsize=12)
plt.title('Distribution of disadvantaged student perceived potentials')#, fontsize=14, fontweight='bold')

# Add a legend to distinguish the datasets
plt.legend(fontsize=10)

# Add a grid for better readability of values
plt.grid(axis='y', linestyle='--', alpha=0.7) # Grid on y-axis is often less cluttered

# Add annotations if specific points are important (optional, but can be powerful)
# Example: Annotate the lower x-limit
# plt.axvline(lower_x_limit, color='green', linestyle=':', linewidth=2, label=f'Lower Limit: {lower_x_limit:.3f}')
# plt.legend(fontsize=10) # Update legend if adding axvline labels

# Tight layout ensures all elements fit within the figure
plt.tight_layout()

# Save the figure with a high resolution for publication
plt.savefig('pps_distribution_plot.png', dpi=300, bbox_inches='tight')
# Or save as a vector graphic for infinite scalability
# plt.savefig('pps_distribution_plot.svg', bbox_inches='tight')

# Display the plot
plt.show()

In [ ]:
def evenly_spaced_take(arr, k):
    indices = np.round(np.linspace(0, len(arr) - 1, k)).astype(int)
    return arr[indices]

In [ ]:
def find_optimal_debiasing_interval(students, budget):
    # we first find the rough area by doing a grid search
    dspps = compute_dspps(students)
    los = evenly_spaced_take(dspps[:int((1-budget)*len(dspps))], 50)
    with_pauc = np.array([(lo, compute_pauc_given_budget(students=students, lo=lo, budget=budget)) for lo in los])
    best_ix = with_pauc[:, 1].argmin()
    lo_bounds = (with_pauc[best_ix-2,0], with_pauc[best_ix+2,0])
    neg_pauc_given_lo = lambda lo: compute_pauc_given_budget(students=students, lo=lo, budget=budget)
    # out = minimize_scalar(neg_pauc_given_lo, compute_bracket(students, budget), tol=1e-10)
    out = minimize_scalar(neg_pauc_given_lo, bounds=lo_bounds, options={"xatol": 1e-5})
    best_lo = out.x
    best_pauc = out.fun
    return best_lo, compute_hi(students, best_lo, budget), best_pauc

In [ ]:
for budget in [.1, .4]:
    print(f"============================")
    print(f"======== {budget=} ========")
    print(f"============================")
    print()
    print(f"QQQQ: Model & PAUC & Optimal & Theoretical & Best red. & Theoretical red. & Factor \\\\")
    th_best_lo, th_best_hi = compute_theorem_interval(p, model_beta, alpha, budget)
    for name, bias_func, best_param in scenarios:
        print(f"===== {name}, {budget=} =====")
        np.random.seed(1)
        bias = partial(bias_func, param=best_param)
        students = generate_students(true_pots, p, bias)

        no_debias_pauc = compute_pauc(students, -1, -1)
#         assert compute_pauc(students, -1, -1) == 0.

        best_lo, best_hi, best_pauc = find_optimal_debiasing_interval(students=students, budget=budget)
        def rep(n, lo, hi, pauc):
            print(f"{name} ({budget=}), {n:>25}: lo={lo:5.3f}, hi={hi:5.3f}, pauc={pauc:5.3f}, prop_pauc={pauc/best_pauc*100:6.2f} %")
        rep("empirical best", lo=best_lo, hi=best_hi, pauc=best_pauc)

        assert compute_pauc(students, -np.inf, np.inf) == 0.
        rep("no debias", lo=0, hi=0, pauc=no_debias_pauc)

        # debias with misspecified best interval
        theoretical_misspecified_pauc = compute_pauc(students, th_best_lo, th_best_hi)
        rep("vs theoretical", lo=th_best_lo, hi=th_best_hi, pauc=theoretical_misspecified_pauc)

        th_emp_hi = compute_hi(students, th_best_lo, budget)
        theoretical_misspecified_pauc_flexhi = compute_pauc(students, th_best_lo, th_emp_hi)
        rep("vs theoretical (flex hi)", lo=th_best_lo, hi=th_emp_hi, pauc=theoretical_misspecified_pauc_flexhi)

        th_emp_lo = compute_lo(students, th_best_hi, budget)
        theoretical_misspecified_pauc_flexlo = compute_pauc(students, th_emp_lo, th_best_hi)
        rep("vs theoretical (flex lo)", lo=th_emp_lo, hi=th_best_hi, pauc=theoretical_misspecified_pauc_flexlo)

        print()
        
        best_red = 1-best_pauc/no_debias_pauc
        th_red = 1-theoretical_misspecified_pauc_flexlo/no_debias_pauc
        
        print()